# 第2回 宿題（発展）: 用語集を LoRA で覚えさせる vs 用語集を検索して渡す

講座の用語集（`data/glossary.json`、58 語）から作った Q&A で `Qwen/Qwen3.5-0.8B` を LoRA 学習し、次の 4 条件を同じ質問で比べる。

| 条件 | モデル | 用語集の渡し方 |
| --- | --- | --- |
| base | 学習前 | 渡さない（モデルの知識だけで答える） |
| base+検索 | 学習前 | 講座の `search_glossary`（`app/tools.py`）を質問文で引き、結果をプロンプトに入れる |
| LoRA | 学習後 | 渡さない |
| LoRA+検索 | 学習後 | 検索結果を入れる |

評価用の質問は 5 種類（すべて学習データに無い文）。

- **seen**: 学習と同じ言い回し（「X」とは何ですか？）。覚えたかどうかの上限を見る
- **paraphrase**: 同じ用語を別の言い回しで聞く（「temperature を下げると出力はどう変わる？」など）
- **reverse**: 定義から用語名を答えさせる（用語名が質問に出てこないので文字列検索が効きにくい）
- **unknown**: 用語集に無い語（学習時の「用語集に無い語」とは別の語）。「用語集にありません」と答えれば正解
- **general**: 用語集と無関係な一般的な質問（システムプロンプトなし）。学習で一般の受け答えが崩れていないかを見る

採点は文字列で行う。定義の要点を 3〜5 個の語群にして、答えに含まれた割合が 2/3 以上なら正解とする（語群の中はどれか 1 つ含まれればよい）。

使うもの: Colab の T4。所要時間は合計 30 分前後（学習 15 分前後）。ライブラリは演習ノートブックと同じ版に固定する。

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定とデータ

`glossary_sft.jsonl` は Codespaces で次のコマンドで作ったもの（233 件 = 用語 58 × 2 テンプレート + 別名での質問 97 + 用語集に無い語 10 × 2）。

```bash
uv run python scripts/make_sft_dataset.py --aliases --negatives > glossary_sft.jsonl
```

Colab の左のファイル一覧に `glossary_sft.jsonl` と `glossary.json` をアップロードしてあればそれを使い、無ければ fork の `lec02` ブランチから取る。

In [ ]:
import json, os, re, time, urllib.request
from pathlib import Path

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 5,
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 160,
    "gen_batch": 8,
    "pass_ratio": 2 / 3,
    "work_dir": "/content/glossary_sft",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)
SRC = "https://raw.githubusercontent.com/Qu0rise/lecture-ai-engineering-2026/lec02"

def fetch(name, url):
    p = Path("/content") / name
    if not p.exists():
        urllib.request.urlretrieve(url, p)
    return p

sft_rows = [json.loads(l) for l in fetch("glossary_sft.jsonl", f"{SRC}/lectures/02/glossary_sft.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
GLOSSARY = json.loads(fetch("glossary.json", f"{SRC}/data/glossary.json").read_text(encoding="utf-8"))["terms"]
TERM = {t["id"]: t for t in GLOSSARY}
print(len(sft_rows), "examples |", len(GLOSSARY), "terms")
print(sft_rows[0])

## 2. 評価セット

用語集から 12 語を選び、seen / paraphrase / reverse を作った。講座に固有の語（宿題の3レベル、lecNN-start タグ、Codespaces secret）と、一般にも知られている語（LoRA、temperature、過学習など）を混ぜている。

`KEYS` が採点に使う語群。定義の中から、言い換えにくい要点を選んだ。

In [ ]:
KEYS = {
    "lora": [("低ランク", "小さな行列", "行列の積"), ("固定", "凍結"), ("アダプタ", "パラメータが少な", "少ないパラメータ", "一部")],
    "hallucination": [("事実と異なる", "誤った", "間違", "存在しない"), ("もっともらし", "それらし"), ("ツール", "根拠", "外部データ", "評価")],
    "temperature": [("ランダム", "ばらつき", "多様"), ("低い", "低く", "下げ"), ("安定", "一貫", "決定的")],
    "codespaces-secret": [("環境変数",), ("fork", "フォーク", "repo", "リポジトリ"), ("再起動",)],
    "homework-levels": [("基礎",), ("実装",), ("発展",), ("どれか1つ", "どれか一つ", "いずれか1つ", "いずれか一つ", "1つ取り組", "ひとつ"), ("採点",)],
    "overfitting": [("学習データ", "訓練データ"), ("未知", "新しい", "汎化"), ("検証",)],
    "qlora": [("量子化", "4ビット", "4bit", "4-bit"), ("LoRA",), ("メモリ",)],
    "mcp": [("標準", "共通", "プロトコル"), ("ツール", "データソース"), ("サーバー", "クライアント")],
    "rate-limit": [("上限", "制限"), ("429",), ("無料枠", "リクエスト数", "回数", "呼び出")],
    "lec-start-tag": [("タグ",), ("開始時点", "演習開始", "始め", "開始"), ("ブランチ",)],
    "harness": [("ループ",), ("ツール",), ("Claude Code", "Pi", "UI", "画面")],
    "human-in-the-loop": [("承認", "確認"), ("人",), ("取り消し", "書き込み", "送信", "requires_approval")],
}
PARAPHRASE = {
    "lora": "LoRAを使うと何がうれしいのですか？仕組みと合わせて教えてください。",
    "hallucination": "生成AIがハルシネーションを起こすって、どういう状態のことですか？",
    "temperature": "temperatureの値を下げると、出力はどう変わりますか？",
    "codespaces-secret": "Codespaces secretにAPIキーを登録すると、プログラムからはどう見えますか？",
    "homework-levels": "AIE2026の宿題の3レベルは、全部やらないといけないのですか？",
    "overfitting": "過学習が起きているかどうかは、どうやって気づけますか？",
    "qlora": "QLoRAは普通のLoRAと何が違うのですか？",
    "mcp": "MCPを使うと、エージェントとツールのつなぎ方はどう変わりますか？",
    "rate-limit": "Gemini APIでレート制限に引っかかると、何が起きますか？",
    "lec-start-tag": "lec03-startというタグは何のためにあるのですか？",
    "harness": "Claude Codeのようなハーネスは、LLMに何を足したものですか？",
    "human-in-the-loop": "エージェントにHuman-in-the-loopを入れるのは、どんなときですか？",
}
REVERSE = [
    ("hallucination", "LLMが事実と異なる内容をもっともらしく生成してしまう現象を何と呼びますか？", ["ハルシネーション", "幻覚", "hallucination"]),
    ("overfitting", "学習データにだけ適合して、未知の入力で性能が落ちる状態を何と言いますか？", ["過学習", "overfitting", "オーバーフィッティング"]),
    ("qlora", "重みを4ビットに量子化したうえでLoRAで学習する手法の名前は何ですか？", ["QLoRA"]),
    ("lec-start-tag", "AIE2026で、各回の演習開始時点のコードに付けてあるGitタグを何と呼びますか？", ["start"]),
    ("mcp", "AIアプリと外部のツールやデータソースをつなぐ、オープンな標準プロトコルは何ですか？", ["MCP", "Model Context Protocol"]),
    ("rate-limit", "一定時間あたりに呼び出せるリクエスト数やトークン数の上限を何と言いますか？", ["レート制限", "rate limit", "レートリミット"]),
]
UNKNOWN = ["RLHF", "知識蒸留", "MoE（Mixture of Experts）", "DPO", "プロンプトインジェクション", "ReAct", "AIE2026の中間試験", "AIE2026の修了証"]
GENERAL = [("日本の首都はどこですか？", ["東京"]), ("3+5はいくつですか？", ["8"]), ("Pythonでリストの長さを求める関数は何ですか？", ["len"]),
           ("英語で「りんご」は何と言いますか？", ["apple"]), ("1週間は何日ですか？", ["7", "七"])]

eval_set = []
for tid in KEYS:
    eval_set.append({"kind": "seen", "id": tid, "q": f"「{TERM[tid]['term']}」とは何ですか？"})
    eval_set.append({"kind": "paraphrase", "id": tid, "q": PARAPHRASE[tid]})
for tid, q, names in REVERSE:
    eval_set.append({"kind": "reverse", "id": tid, "q": q, "names": names})
for w in UNKNOWN:
    eval_set.append({"kind": "unknown", "id": None, "q": f"「{w}」とは何ですか？"})

# 評価の質問が学習データにそのまま入っていないことを確かめる（seen は「学習と同じ言い回し」なので入っている）
train_qs = {r["messages"][0]["content"] for r in sft_rows}
leak = [e["q"] for e in eval_set if e["kind"] != "seen" and e["q"] in train_qs]
assert not leak, leak
assert all(e["q"] in train_qs for e in eval_set if e["kind"] == "seen")
assert not any(w in train_qs for w in UNKNOWN)
print({k: sum(e["kind"] == k for e in eval_set) for k in ["seen", "paraphrase", "reverse", "unknown"]}, "+ general", len(GENERAL))

## 3. 検索（講座の `search_glossary` と同じ処理）

第3回の `app/tools.py` の `search_glossary` をそのまま写した。エージェントならモデルが検索語を決めるが、ここでは質問文をそのまま検索語にする。部分一致は「用語名（3 文字以上）が質問文に含まれるか」で当たる。

In [ ]:
MAX_RESULTS = 3

def _normalize(text):
    return text.casefold().replace(" ", "").replace("　", "")

def _names(entry):
    return [_normalize(entry["term"]), _normalize(entry["id"])] + [_normalize(a) for a in entry["aliases"]]

def search_glossary_entries(query):
    q = _normalize(query)
    exact = [e for e in GLOSSARY if q in _names(e)]
    partial = [e for e in GLOSSARY if e not in exact and any(q in n or (len(n) >= 3 and n in q) for n in _names(e))]
    in_def = [e for e in GLOSSARY if e not in exact and e not in partial and q in _normalize(e["definition"])]
    return (exact + partial + in_def)[:MAX_RESULTS]

def search_glossary(query):
    hits = search_glossary_entries(query)
    if not hits:
        return f"用語集に「{query}」は見つかりませんでした。"
    return json.dumps([{"term": e["term"], "lecture": e["lecture"], "definition": e["definition"], "related": e["related"]} for e in hits], ensure_ascii=False)

for e in eval_set:
    e["retrieved"] = [h["id"] for h in search_glossary_entries(e["q"])]
    e["retrieval_hit"] = (e["id"] in e["retrieved"]) if e["id"] else (len(e["retrieved"]) == 0)
import pandas as pd
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(eval_set).groupby("kind")["retrieval_hit"].agg(["sum", "count"]))
display(pd.DataFrame([e for e in eval_set if not e["retrieval_hit"]])[["kind", "q", "id", "retrieved"]])

## 4. モデルと生成

システムプロンプトは全条件で同じにする（学習データにも同じものを付ける）。生成は貪欲法で、8 問ずつまとめて生成する。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.manual_seed(CONFIG["seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)

SYSTEM = ("あなたはAIE2026講座の用語集の案内係です。講座の用語集に載っている用語について、用語集の定義に沿って簡潔に答えてください。"
          "用語集に無い用語を聞かれたら「この用語は講座の用語集にありません。」とだけ答えてください。")

def messages(q, use_search=False, system=True):
    user = f"用語集の検索結果:\n{search_glossary(q)}\n\n質問: {q}" if use_search else q
    return ([{"role": "system", "content": SYSTEM}] if system else []) + [{"role": "user", "content": user}]

def generate_batch(m, msgs_list):
    outs = []
    tok.padding_side = "left"
    for i in range(0, len(msgs_list), CONFIG["gen_batch"]):
        prompts = [tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=True, enable_thinking=False) for ms in msgs_list[i:i + CONFIG["gen_batch"]]]
        enc = tok(prompts, return_tensors="pt", padding=True).to(m.device)
        with torch.no_grad():
            out = m.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], do_sample=False, pad_token_id=tok.pad_token_id)
        outs += [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
    tok.padding_side = "right"
    return outs

## 5. 採点

In [ ]:
def norm(s):
    return _normalize(s)

def refused(a):
    a = norm(a)
    return any(p in a for p in ["用語集にありません", "用語集に載っていません", "用語集には載っていません", "用語集には記載", "用語集に記載がありません", "見つかりませんでした"])

def score(e, a):
    if e["kind"] == "unknown":
        return {"ok": refused(a), "refused": refused(a), "key_ratio": None}
    if e["kind"] == "reverse":
        ok = any(norm(n) in norm(a) for n in e["names"]) and not refused(a)
        return {"ok": ok, "refused": refused(a), "key_ratio": None}
    groups = KEYS[e["id"]]
    hit = sum(any(norm(w) in norm(a) for w in g) for g in groups)
    r = hit / len(groups)
    return {"ok": (r >= CONFIG["pass_ratio"]) and not refused(a), "refused": refused(a), "key_ratio": round(r, 2)}

def run(m, label, use_search):
    t0 = time.time()
    answers = generate_batch(m, [messages(e["q"], use_search) for e in eval_set])
    rows = [{**{k: e[k] for k in ["kind", "id", "q"]}, "answer": a, **score(e, a)} for e, a in zip(eval_set, answers)]
    summ = {"label": label, "seconds": round(time.time() - t0, 1)}
    for kind in ["seen", "paraphrase", "reverse", "unknown"]:
        rs = [r for r in rows if r["kind"] == kind]
        summ[kind] = f"{sum(r['ok'] for r in rs)}/{len(rs)}"
    known = [r for r in rows if r["kind"] != "unknown"]
    summ["over_refusal"] = f"{sum(r['refused'] for r in known)}/{len(known)}"   # 用語集にある語なのに「ありません」と答えた
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summ, "rows": rows}, ensure_ascii=False, indent=1), encoding="utf-8")
    print(json.dumps(summ, ensure_ascii=False))
    return summ, rows

def run_general(m, label):
    answers = generate_batch(m, [messages(q, system=False) for q, _ in GENERAL])
    rows = [{"q": q, "answer": a, "ok": any(x.casefold() in a.casefold() for x in exp)} for (q, exp), a in zip(GENERAL, answers)]
    print(label, f"general {sum(r['ok'] for r in rows)}/{len(rows)}")
    return rows

results, details = {}, {}
for label, use_search in [("base", False), ("base+search", True)]:
    results[label], details[label] = run(model, label, use_search)
general = {"base": run_general(model, "base")}

## 6. LoRA 学習

演習ノートブックのセクション 4 と同じ設定（r=16、lr=2e-4、バッチ 4 × 勾配蓄積 2、cosine）。データが 233 件あるので、エポックは 5 にした（約 150 ステップ）。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list([{"messages": [{"role": "system", "content": SYSTEM}] + r["messages"]} for r in sft_rows])
model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_strategy="no",
                max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
print(f"学習時間 {TRAIN_SECONDS} 秒 | ステップ {trainer.state.global_step} | loss {losses[0]:.3f} -> {losses[-1]:.3f}")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 7. 学習後の評価と比較

In [ ]:
for label, use_search in [("lora", False), ("lora+search", True)]:
    results[label], details[label] = run(tuned, label, use_search)
general["lora"] = run_general(tuned, "lora")

table = pd.DataFrame(results.values()).set_index("label")
display(table)

# 質問ごとの答え（4 条件を横に並べる）
wide = pd.DataFrame([{"kind": e["kind"], "質問": e["q"],
                      **{lab: ("○ " if details[lab][i]["ok"] else "× ") + details[lab][i]["answer"][:90] for lab in details}}
                     for i, e in enumerate(eval_set)])
display(wide)
display(pd.DataFrame([{"質問": q, "base": b["answer"][:60], "lora": l["answer"][:60]} for (q, _), b, l in zip(GENERAL, general["base"], general["lora"])]))

## 8. result.json

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k != "work_dir"},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "data": {"train_examples": len(sft_rows), "eval": {k: sum(e["kind"] == k for e in eval_set) for k in ["seen", "paraphrase", "reverse", "unknown"]},
             "general": len(GENERAL)},
    "retrieval_hit": {k: f"{sum(e['retrieval_hit'] for e in eval_set if e['kind'] == k)}/{sum(e['kind'] == k for e in eval_set)}" for k in ["seen", "paraphrase", "reverse", "unknown"]},
    "train": {"seconds": TRAIN_SECONDS, "steps": trainer.state.global_step, "loss_first": round(losses[0], 3), "loss_last": round(losses[-1], 3)},
    "results": results,
    "general": {k: f"{sum(r['ok'] for r in v)}/{len(v)}" for k, v in general.items()},
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "details.json").write_text(json.dumps({"eval": details, "general": general}, ensure_ascii=False, indent=1), encoding="utf-8")
print("RESULT_JSON_BEGIN"); print(json.dumps(result, ensure_ascii=False)); print("RESULT_JSON_END")